In [10]:
import pandas as pd
import re
import gensim
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import word_tokenize

In [11]:
docs = ['This is the first document.', 'This document is the second document.', 'And this is the third one.', 'Is this the first document?']
tfidf = TfidfVectorizer()
vectors = tfidf.fit_transform(docs)
pd.DataFrame(cosine_similarity(vectors), index=docs, columns=docs)

,This is the first document.,This document is the second document.,And this is the third one.,Is this the first document?
This is the first document.,1.000000,0.646926,0.307772,1.000000
This document is the second document.,0.646926,1.000000,0.225240,0.646926
And this is the third one.,0.307772,0.225240,1.000000,0.307772
Is this the first document?,1.000000,0.646926,0.307772,1.000000


In [12]:
docs = ['data science is one of the most important fields of science', 'this is one of the best data science courses', 'data scientists analyze data']
tfidf = TfidfVectorizer()
result = tfidf.fit_transform(docs)
df = pd.DataFrame(result.toarray(), columns=tfidf.get_feature_names_out(), index=docs)
df.round(3)

,analyze,best,courses,data,fields,important,is,most,of,one,science,scientists,the,this
data science is one of the most important fields of science,0.000,0.0,0.0,0.190,0.321,0.321,0.244,0.321,0.488,0.244,0.488,0.000,0.244,0.0
this is one of the best data science courses,0.000,0.4,0.4,0.236,0.000,0.000,0.304,0.000,0.304,0.304,0.304,0.000,0.304,0.4
data scientists analyze data,0.543,0.0,0.0,0.641,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.543,0.000,0.0


In [13]:
for doc, row in df.iterrows():
    print(doc, '->', row.nlargest(3).round(3).to_dict())

data science is one of the most important fields of science -> {'of': 0.488, 'science': 0.488, 'fields': 0.321}
this is one of the best data science courses -> {'best': 0.4, 'courses': 0.4, 'this': 0.4}
data scientists analyze data -> {'data': 0.641, 'analyze': 0.543, 'scientists': 0.543}


In [14]:
df = pd.read_csv('simpsons_script_lines.csv')
df = df.dropna(subset=['spoken_words'])
df.head()

,raw_character_text,spoken_words
0,Miss Hoover,"No, actually, it was a little of both. Sometim..."
1,Lisa Simpson,Where's Mr. Bergstrom?
2,Miss Hoover,I don't know. Although I'd sure like to talk t...
3,Lisa Simpson,That life is worth living.
4,Edna Krabappel-Flanders,The polls will be open from now until the end ...


In [15]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", '', text)
    text = re.sub(r"[)(,”“.’$-]", '', text)
    return text

tokens = [word_tokenize(clean_text(line)) for line in df['spoken_words']]
tokens[:3]

[['no',
  'actually',
  'it',
  'was',
  'a',
  'little',
  'of',
  'both',
  'sometimes',
  'when',
  'a',
  'disease',
  'is',
  'in',
  'all',
  'the',
  'magazines',
  'and',
  'all',
  'the',
  'news',
  'shows',
  'it',
  "'s",
  'only',
  'natural',
  'that',
  'you',
  'think',
  'you',
  'have',
  'it'],
 ['where', "'s", 'mr', 'bergstrom', '?'],
 ['i',
  'do',
  "n't",
  'know',
  'although',
  'i',
  "'d",
  'sure',
  'like',
  'to',
  'talk',
  'to',
  'him',
  'he',
  'did',
  "n't",
  'touch',
  'my',
  'lesson',
  'plan',
  'what',
  'did',
  'he',
  'teach',
  'you',
  '?']]

In [16]:
Skip_gram_model = gensim.models.Word2Vec(tokens, min_count=1, vector_size=100, window=5, sg=1)
print(len(Skip_gram_model.wv))

44175


In [17]:
for word in ['homer', 'marge', 'bart']:
    print(word, '->', Skip_gram_model.wv.most_similar(word))
    print()

homer -> [('abe', 0.8719659447669983), ('marge', 0.859354555606842), ('bart', 0.8326413631439209), ('eliza', 0.8208022117614746), ('bartholomew', 0.8153685331344604), ('grampa', 0.8146928548812866), ('ned', 0.8040746450424194), ('waylon', 0.8039629459381104), ('apu', 0.8038920760154724), ('sweetheart', 0.7970868349075317)]

marge -> [('homer', 0.8593546152114868), ('abe', 0.8570330739021301), ('becky', 0.8251453042030334), ('sweetie', 0.8169563412666321), ('sweetheart', 0.8157606720924377), ('honey', 0.8052242398262024), ('lisa', 0.804006814956665), ('eliza', 0.7998676300048828), ('midge', 0.7961438894271851), ('lurleen', 0.7925573587417603)]

bart -> [('milhouse', 0.8653382062911987), ('lisa', 0.8580104112625122), ('grampa', 0.8470678925514221), ('eliza', 0.8465099930763245), ('sweetie', 0.8420118689537048), ('jessica', 0.8393140435218811), ('abe', 0.8388946652412415), ('homer', 0.8326413035392761), ('sweetheart', 0.8295297622680664), ('laddie', 0.8274993300437927)]



In [18]:
print(Skip_gram_model.wv.doesnt_match(['jimbo', 'milhouse', 'kearney']))
print(Skip_gram_model.wv.doesnt_match(['nelson', 'bart', 'milhouse']))
print(Skip_gram_model.wv.doesnt_match(['homer', 'patty', 'selma']))

milhouse
nelson
homer
